# Exercício 2 — Duplicatas

Análise da planilha Transactions do conjunto KPMG.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

arquivo = Path("dados/KPMG_VI_New_raw_data_update_final.xlsx")
if not arquivo.exists():
    arquivo = Path("KPMG_VI_New_raw_data_update_final.xlsx")
transacoes = pd.read_excel(arquivo, sheet_name="Transactions", header=1)
transacoes.head()

## Cabeçalho e tipos

In [ ]:
print(transacoes.shape)
transacoes.dtypes.to_frame("tipo")

## Duplicatas exatas

In [ ]:
duplicatas_exatas = transacoes.duplicated(keep=False)
print("Linhas em duplicatas exatas:", int(duplicatas_exatas.sum()))
transacoes[duplicatas_exatas].head()

## Duplicatas por subconjunto

In [ ]:
candidatas_id = [c for c in ["transaction_id", "customer_id", "product_id", "transaction_date"] if c in transacoes.columns]
chave = [c for c in ["transaction_id"] if c in transacoes.columns]
if not chave:
    chave = candidatas_id
print("Chave escolhida:", chave)
if chave:
    duplicatas_chave = transacoes.duplicated(subset=chave, keep=False)
    print("Linhas repetidas pela chave:", int(duplicatas_chave.sum()))
    transacoes[duplicatas_chave].sort_values(chave).head()

## Unicidade dos identificadores

In [ ]:
unicidade = pd.DataFrame({"nulos": transacoes.isna().sum(), "valores_distintos": transacoes.nunique(dropna=True), "linhas": len(transacoes)})
unicidade[unicidade.index.isin(candidatas_id)]

## Estratégia e impacto

In [ ]:
df_transacoes_limpo = transacoes.copy()
antes = len(df_transacoes_limpo)
if chave:
    df_transacoes_limpo = df_transacoes_limpo.drop_duplicates(subset=chave, keep="last")
depois = len(df_transacoes_limpo)
coluna_preco = next((c for c in ["list_price", "list price", "listprice"] if c in transacoes.columns), None)
impacto = pd.DataFrame({"linhas": [antes, depois], "registros_afetados": [0, antes-depois]})
if coluna_preco:
    impacto["soma_list_price"] = [pd.to_numeric(transacoes[coluna_preco], errors="coerce").sum(), pd.to_numeric(df_transacoes_limpo[coluna_preco], errors="coerce").sum()]
impacto

## Desafio — textos aproximados

In [ ]:
try:
    from rapidfuzz import process, fuzz
    texto = transacoes.select_dtypes(include="object").columns[0]
    valores = transacoes[texto].dropna().astype(str).drop_duplicates().tolist()
    pares = []
    for valor in valores[:300]:
        similares = process.extract(valor, valores, scorer=fuzz.token_sort_ratio, limit=3, score_cutoff=85)
        pares.extend([(valor, outro, pontuacao) for outro, pontuacao, _ in similares if outro != valor])
    pd.DataFrame(pares, columns=["valor", "similar", "pontuacao"]).drop_duplicates().head(20)
except ImportError:
    print("Instale rapidfuzz para executar o desafio")

## CustomerDemographic — categorias de gênero

In [ ]:
demografico = pd.read_excel(arquivo, sheet_name="CustomerDemographic", header=1)
coluna_genero = next((c for c in demografico.columns if str(c).lower() in ["gender", "genero"]), None)
if coluna_genero:
    demografico[coluna_genero].astype("string").str.strip().str.casefold().value_counts(dropna=False)
else:
    demografico.head()

## Respostas de reflexão

Zero duplicatas exatas só significa que não há linhas completamente iguais. O mesmo evento pode aparecer com um campo alterado, com espaços, com tipos diferentes ou com outra chave de identificação.

Duas linhas idênticas podem não ser um erro se representarem duas compras legítimas feitas pelo mesmo cliente, no mesmo produto e com os mesmos valores. A data, o identificador da transação ou outro campo do contexto precisa distinguir esses eventos.

A escolha da chave é uma decisão de negócio porque define o que a empresa considera uma transação. Essa definição pertence ao Entendimento do Negócio do CRISP-DM: remover registros que parecem iguais tecnicamente pode apagar eventos reais, enquanto manter repetições pode inflar faturamento e ticket médio.